# Chaldene Token API — Developer Tutorial

This notebook is a hands-on tutorial for **JupyterLab extension developers** who want to interact with Chaldene visual programming (VP) cells programmatically.

The **Chaldene Token API** (`IChaldeneService`) lets an external JupyterLab plugin:

| Capability | Method / Signal |
|---|---|
| Discover live VP cells | `isCellReady`, `getReadyCellIds`, `cellReady` signal |
| Read a cell's node graph | `getGraph` |
| Replace a cell's node graph | `setGraph` |
| Update a single node input | `setInputValue` |
| Trigger cell execution | `run` |
| React to graph changes | `graphChanged` signal |
| React to cell teardown | `cellDisposed` signal |

The API is a **JupyterLab frontend (TypeScript) API**. This notebook presents the TypeScript extension code you need to write alongside runnable Python cells that explore the underlying graph data model.

---
**Repository**: `src/tokens.ts` (interface + Token), `src/ChaldeneService.ts` (implementation)


## Prerequisites

- JupyterLab 4 with the Chaldene extension installed
- A notebook open with at least one VP (visual node) cell
- Familiarity with [JupyterLab extension development](https://jupyterlab.readthedocs.io/en/stable/extension/extension_tutorial.html) (TypeScript, Lumino plugins)

To run the Python cells in this notebook, no extra packages beyond a standard JupyterLab kernel are required.

---
## 1. Core Concepts

### 1.1 VP Cells and Cell IDs

Every visual programming cell is a standard JupyterLab code cell whose source is a JSON-serialised node graph. When the cell is rendered, Chaldene mounts a React canvas (`VPWidget`) on top of it and assigns a **cell ID** — the string returned by `cell.model.sharedModel.getId()`.

```
Cell ID example: 'c3d1a2b4-4e5f-6789-abcd-ef0123456789'
```

The cell ID is the key used for every API call. It is stable for the lifetime of the cell but changes if the notebook is duplicated or the cell is recreated.

### 1.2 Lumino Signals

Chaldene uses [Lumino Signals](https://lumino.readthedocs.io/en/stable/api/modules/signaling.html) (`@lumino/signaling`) for reactive notifications. A signal is similar to an event emitter, but typed and owned by a sender object.

```typescript
// Connect a handler
service.cellReady.connect((sender, cellId) => {
  console.log('cell is ready:', cellId);
});

// Disconnect when done (important — prevents memory leaks)
service.cellReady.disconnect(myHandler);
```

All three signals on `IChaldeneService` fire synchronously, so a connected handler runs immediately before `register` / `deregister` / `updateGraph` returns.

---
## 2. Obtaining the Service in Your Extension

The service is provided via the standard JupyterLab [plugin token](https://jupyterlab.readthedocs.io/en/stable/extension/extension_dev.html#tokens) pattern. Your plugin declares it as a `requires` dependency and receives it as a parameter to `activate`.

### 2.1 Importing the token

Add `chaldene` as a peer dependency in your extension's `package.json`:

```json
{
  "peerDependencies": {
    "chaldene": "^1.0.0"
  }
}
```

Then import only from `chaldene/tokens` — this file has no `@xyflow/react` dependency and is safe to import without pulling in the full VP canvas bundle:

```typescript
import { IChaldeneService } from 'chaldene/tokens';
```

### 2.2 Plugin descriptor

Declare your plugin with `requires: [IChaldeneService]`:

```typescript
import {
  type JupyterFrontEnd,
  type JupyterFrontEndPlugin
} from '@jupyterlab/application';
import { IChaldeneService } from 'chaldene/tokens';

const myPlugin: JupyterFrontEndPlugin<void> = {
  id: 'my-extension:plugin',
  autoStart: true,
  requires: [IChaldeneService],        // <-- declare dependency
  activate: (app: JupyterFrontEnd, chaldene: IChaldeneService) => {
    // chaldene is ready to use immediately
    console.log('Ready cells:', chaldene.getReadyCellIds());
  }
};

export default myPlugin;
```

> **Why `requires` instead of `optional`?** If your extension only makes sense when Chaldene is present, use `requires`. If it can operate without Chaldene and degrades gracefully, use `optional` and guard every call with `if (chaldene) { ... }`.

---
## 3. Discovery — Finding Live VP Cells

A VP cell becomes *ready* once its React canvas has mounted and wired itself to the service. This happens asynchronously after the notebook renders.

### 3.1 Synchronous query

If the cells are already rendered when your plugin activates, you can query synchronously:

```typescript
// Is a specific cell ready?
const ready: boolean = chaldene.isCellReady('c3d1a2b4-...');

// All currently ready cells
const ids: string[] = chaldene.getReadyCellIds();
console.log(`${ids.length} VP cells are live`);
```

### 3.2 Reactive: `cellReady` signal

Subscribe to `cellReady` to be notified whenever a new VP cell finishes mounting:

```typescript
chaldene.cellReady.connect((sender, cellId) => {
  console.log('New VP cell ready:', cellId);
  // Apply any pending state to this cell
  const graph = buildDefaultGraph();
  chaldene.setGraph(cellId, graph);
});
```

### 3.3 Reactive: `cellDisposed` signal

Subscribe to `cellDisposed` to clean up any state you hold for a cell:

```typescript
chaldene.cellDisposed.connect((sender, cellId) => {
  myStateStore.delete(cellId);   // release references
});
```

> `cellDisposed` fires when the user **deletes** the cell or when the notebook panel is closed.

---
## 4. The `IGraph` Data Model

Before reading or writing graphs you need to understand the data model. A graph is a plain JSON-serialisable object:

```typescript
interface IGraph {
  nodes: IGraphNode[];
  edges: IGraphEdge[];
}

interface IGraphNode {
  id: string;                           // unique within the graph, e.g. '0'
  type?: string;                         // node spec name, e.g. 'read_image'
  position: { x: number; y: number };   // canvas position
  data: Record<string, unknown>;         // spec data (see below)
  selected?: boolean;
  width?: number;
  height?: number;
}

interface IGraphEdge {
  id: string;                            // unique within the graph, e.g. '0'
  source: string;                        // source node ID
  sourceHandle: string | null;           // source handle ID, e.g. 'out0'
  target: string;                        // target node ID
  targetHandle: string | null;           // target handle ID, e.g. 'in0'
  selected?: boolean;
}
```

### 4.1 The `data` field

The `data` field of a node carries the node's specification metadata:

| Key | Type | Description |
|---|---|---|
| `specName` | `string` | Name of the node spec (must match a registered spec) |
| `displayLabel` | `string` | Human-readable node name |
| `description` | `string` | Tooltip text |
| `inputs` | `IHandle[]` | Input handles with current values |
| `outputs` | `IHandle[]` | Output handles |

Each handle looks like:

```typescript
interface IHandle {
  id: string;              // e.g. 'in0', 'out0'
  name: string;            // e.g. 'path', 'image'
  type?: string | string[];  // e.g. 'string', 'image', ['image', 'binary image']
  displayLabel?: string;
  description?: string;
  defaultValue?: any;      // current value for input handles
  widget?: { type: string; [key: string]: any };
}
```

> **`editorContext` is stripped automatically.** The API always removes the internal `editorContext` reference from `data` before returning a graph to you, and before firing `graphChanged`. You never need to set or clear it.

In [ ]:
import json

# This Python cell builds example IGraph objects that match the TypeScript
# interfaces above. Run it to inspect the expected structure.

def make_handle(handle_id, name, h_type=None, label=None, default_value=None, widget=None):
    h = {"id": handle_id, "name": name}
    if h_type:        h["type"] = h_type
    if label:         h["displayLabel"] = label
    if default_value is not None: h["defaultValue"] = default_value
    if widget:        h["widget"] = widget
    return h

def make_node(node_id, spec_name, display_label, inputs, outputs, x=100, y=100):
    return {
        "id": str(node_id),
        "type": spec_name,
        "position": {"x": x, "y": y},
        "data": {
            "specName": spec_name,
            "displayLabel": display_label,
            "inputs": inputs,
            "outputs": outputs
        }
    }

def make_edge(edge_id, src_node, src_handle, tgt_node, tgt_handle):
    return {
        "id": str(edge_id),
        "source": str(src_node),
        "sourceHandle": src_handle,
        "target": str(tgt_node),
        "targetHandle": tgt_handle
    }

# ── Minimal single-node graph ───────────────────────────────────────────────
minimal_graph = {
    "nodes": [
        make_node(
            node_id=0,
            spec_name="read_image",
            display_label="read image",
            inputs=[
                make_handle("in0", "path", "string", "file",
                            default_value="/data/sample.png",
                            widget={"type": "FileInputFromServer",
                                    "extensions": [".jpg", ".jpeg", ".png"]}),
                make_handle("in1", "mode", label="mode", default_value="GRAY",
                            widget={"type": "Dropdown", "options": ["GRAY", "RGB"]})
            ],
            outputs=[
                make_handle("out0", "image", "image", "image")
            ]
        )
    ],
    "edges": []
}

print("=== Minimal IGraph (1 node, 0 edges) ===")
print(json.dumps(minimal_graph, indent=2))

In [ ]:
# Two-node pipeline: read_image → threshold
pipeline_graph = {
    "nodes": [
        make_node(
            node_id=0,
            spec_name="read_image",
            display_label="read image",
            inputs=[
                make_handle("in0", "path", "string", "file",
                            default_value="/data/sample.png",
                            widget={"type": "FileInputFromServer",
                                    "extensions": [".jpg", ".jpeg", ".png"]}),
                make_handle("in1", "mode", label="mode", default_value="GRAY",
                            widget={"type": "Dropdown", "options": ["GRAY", "RGB"]})
            ],
            outputs=[make_handle("out0", "image", "image", "image")],
            x=100, y=100
        ),
        make_node(
            node_id=1,
            spec_name="threshold",
            display_label="threshold",
            inputs=[
                make_handle("in0", "image", "image", "image"),
                make_handle("in1", "thresh", label="threshold", default_value=0.5,
                            widget={"type": "Float"})
            ],
            outputs=[make_handle("out0", "binary_image", "binary image", "binary image")],
            x=500, y=100
        )
    ],
    "edges": [
        make_edge(edge_id=0,
                  src_node=0, src_handle="out0",
                  tgt_node=1, tgt_handle="in0")
    ]
}

print("=== Two-node pipeline IGraph ===")
print(json.dumps(pipeline_graph, indent=2))

> **Handle IDs follow the pattern `in0`, `in1`, … `out0`, `out1`, …`**  
> These are assigned by the node spec in order of declaration. Always copy the handle IDs from the node spec or from a `getGraph` call — do not assume numeric order holds for all node types.

---
## 5. Reading Graph State — `getGraph`

`getGraph(cellId)` returns the current graph of a cell as an `IGraph` object, or `undefined` if the cell is not ready or has no graph yet.

```typescript
const graph = chaldene.getGraph(cellId);

if (!graph) {
  console.warn('Cell not ready or empty');
  return;
}

console.log(`Graph has ${graph.nodes.length} nodes, ${graph.edges.length} edges`);

// Read a specific node's input value
const readNode = graph.nodes.find(n => n.type === 'read_image');
const pathHandle = (readNode?.data.inputs as any[])?.find(h => h.id === 'in0');
console.log('Current file path:', pathHandle?.defaultValue);
```

### Key properties of the returned graph

- The graph is a **snapshot copy** — modifying it does not affect the live cell.
- `editorContext` is always absent from `node.data` (stripped before returning).
- Output handle values (thumbnails, image previews) may be large base64 strings — avoid logging them in full.

In [ ]:
# Reading a VP cell's graph from Python (offline inspection)
#
# VP cells store their graph as JSON in the cell source.
# This cell shows how to parse it from the saved notebook file.
# At runtime in JupyterLab, use chaldene.getGraph(cellId) instead.

import json, pathlib

# Point this at any Chaldene notebook to inspect its VP cell graphs.
NOTEBOOK_PATH = pathlib.Path("use_cases/1. grain_size_measurement/p5_grain_size_measurement.ipynb")

if NOTEBOOK_PATH.exists():
    nb = json.loads(NOTEBOOK_PATH.read_text(encoding='utf-8'))
    vp_cells = [
        cell for cell in nb['cells']
        if cell.get('metadata', {}).get('code type') == 'visual code'
    ]
    print(f"Found {len(vp_cells)} VP cell(s) in {NOTEBOOK_PATH.name}")

    for i, cell in enumerate(vp_cells):
        src = ''.join(cell['source'])
        graph = json.loads(src)
        node_types = [n['type'] for n in graph['nodes']]
        print(f"\n  Cell {i}: {len(graph['nodes'])} nodes, {len(graph['edges'])} edges")
        print(f"  Node types: {node_types}")
else:
    print("Notebook not found — adjust NOTEBOOK_PATH to point to a Chaldene .ipynb file.")

---
## 6. Writing a Complete Graph — `setGraph`

`setGraph(cellId, graph)` replaces the entire node graph of a cell. It returns `true` on success and `false` if the cell is not ready.

```typescript
const graph: IGraph = {
  nodes: [
    {
      id: '0',
      type: 'read_image',
      position: { x: 100, y: 100 },
      data: {
        specName: 'read_image',
        displayLabel: 'read image',
        inputs: [
          { id: 'in0', name: 'path', type: 'string',
            displayLabel: 'file', defaultValue: '/data/sample.png',
            widget: { type: 'FileInputFromServer', extensions: ['.jpg', '.png'] } },
          { id: 'in1', name: 'mode', displayLabel: 'mode', defaultValue: 'GRAY',
            widget: { type: 'Dropdown', options: ['GRAY', 'RGB'] } }
        ],
        outputs: [
          { id: 'out0', name: 'image', type: 'image', displayLabel: 'image' }
        ]
      }
    }
  ],
  edges: []
};

const ok = chaldene.setGraph(cellId, graph);
console.log('Graph applied:', ok);   // true if cell was ready
```

### What `setGraph` does internally

1. Calls `EditorContext.newGraphInput(graph)` on the target cell.
2. `newGraphInput` reseeds the node/edge ID counters from the graph's existing IDs (so the next auto-generated ID won't collide).
3. Dispatches a React state update — the canvas re-renders with the new graph **without unmounting**.
4. Does **not** set `blockTriggerRunCode = false`, so live auto-execution is not armed.

> **Why no live execution?** VP cells only auto-execute when the user interactively edits them. A programmatic `setGraph` should not silently trigger a kernel run — call `chaldene.run(cellId)` explicitly if you want execution.

In [ ]:
# Build a complete 3-node image processing pipeline.
# This is the same structure your TypeScript code would send to setGraph.

import json

def make_handle(handle_id, name, h_type=None, label=None, default_value=None, widget=None):
    h = {"id": handle_id, "name": name}
    if h_type:                    h["type"] = h_type
    if label:                     h["displayLabel"] = label
    if default_value is not None: h["defaultValue"] = default_value
    if widget:                    h["widget"] = widget
    return h

def make_node(node_id, spec_name, display_label, inputs, outputs, x=100, y=100):
    return {
        "id": str(node_id),
        "type": spec_name,
        "position": {"x": x, "y": y},
        "data": {
            "specName": spec_name,
            "displayLabel": display_label,
            "inputs": inputs,
            "outputs": outputs
        }
    }

def make_edge(edge_id, src_node, src_handle, tgt_node, tgt_handle):
    return {
        "id": str(edge_id),
        "source": str(src_node),
        "sourceHandle": src_handle,
        "target": str(tgt_node),
        "targetHandle": tgt_handle
    }

# ── 3-node pipeline: read → denoise → threshold ──────────────────────────────
graph = {
    "nodes": [
        make_node(0, "read_image", "read image",
            inputs=[
                make_handle("in0", "path", "string", "file",
                            default_value="/data/cells.png",
                            widget={"type": "FileInputFromServer",
                                    "extensions": [".jpg", ".jpeg", ".png"]}),
                make_handle("in1", "mode", label="mode", default_value="GRAY",
                            widget={"type": "Dropdown", "options": ["GRAY", "RGB"]})
            ],
            outputs=[make_handle("out0", "image", "image", "image")],
            x=100, y=150),

        make_node(1, "denoise_bilateral", "denoise bilateral",
            inputs=[make_handle("in0", "image", "image", "image")],
            outputs=[make_handle("out0", "outputImage", "image", "image")],
            x=500, y=150),

        make_node(2, "threshold", "threshold",
            inputs=[
                make_handle("in0", "image", "image", "image"),
                make_handle("in1", "thresh", label="threshold", default_value=0.5,
                            widget={"type": "Float"})
            ],
            outputs=[make_handle("out0", "binary_image", "binary image", "binary image")],
            x=900, y=150)
    ],
    "edges": [
        make_edge(0, src_node=0, src_handle="out0", tgt_node=1, tgt_handle="in0"),
        make_edge(1, src_node=1, src_handle="out0", tgt_node=2, tgt_handle="in0")
    ]
}

print("=== 3-node pipeline IGraph ===")
print(f"  Nodes : {len(graph['nodes'])}")
print(f"  Edges : {len(graph['edges'])}")
for n in graph['nodes']:
    print(f"  [{n['id']}] {n['data']['specName']} "
          f"  inputs={[h['id'] for h in n['data']['inputs']]} "
          f"  outputs={[h['id'] for h in n['data']['outputs']]}")
for e in graph['edges']:
    print(f"  edge {e['id']}: {e['source']}.{e['sourceHandle']} → {e['target']}.{e['targetHandle']}")

# This is the object you would pass to chaldene.setGraph(cellId, graph)
# in TypeScript (the field names and structure are identical).

---
## 7. Fine-grained Input Updates — `setInputValue`

`setInputValue(cellId, nodeId, handleId, value)` changes a single input handle's value without replacing the entire graph. Use this for parameter sweeps or when the graph structure is fixed and only a value changes.

```typescript
// Change the threshold value on node '2', handle 'in1'
chaldene.setInputValue(cellId, '2', 'in1', 0.75);

// Change the file path on node '0', handle 'in0'
chaldene.setInputValue(cellId, '0', 'in0', '/data/new_image.png');

// Values can be strings, numbers, arrays, or plain objects
chaldene.setInputValue(cellId, '3', 'in2', [0, 0, 256, 256]);  // crop area
chaldene.setInputValue(cellId, '4', 'in0', { sigma: 1.5, mode: 'reflect' });
```

### When to use `setInputValue` vs `setGraph`

| Situation | API to use |
|---|---|
| Graph topology must change (add/remove nodes or edges) | `setGraph` |
| Only parameter values change, structure is stable | `setInputValue` |
| Rebuilding a graph from an external specification | `setGraph` |
| Parameter sweep over a single variable | `setInputValue` in a loop |

> **Node IDs are strings.** Even if your IDs look numeric (`'0'`, `'1'`), pass them as strings. The API signature accepts `string` for both `nodeId` and `handleId`.

---
## 8. Triggering Execution — `run`

`run(cellId)` executes the VP cell — identical to clicking the run button or pressing Shift+Enter while focused on the cell.

```typescript
// Update a parameter, then run
chaldene.setInputValue(cellId, '2', 'in1', newThreshold);
chaldene.run(cellId);
```

### Execution guard: `blockTriggerRunCode`

VP cells have an internal flag called `blockTriggerRunCode` that prevents accidental execution on load. This flag is:

- **`true` (blocked)** initially and after `setGraph` — safe state, no auto-run
- **`false` (armed)** only after the user interactively edits the graph via the UI

Calling `run(cellId)` bypasses this flag — it always executes. The flag only governs *automatic* live-execution triggered by graph edits.

```typescript
// Pattern: set graph, then explicitly decide whether to run
chaldene.setGraph(cellId, graph);   // blockTriggerRunCode stays true

if (shouldRunImmediately) {
  chaldene.run(cellId);             // explicit run — always safe
}
```

### Handling async results

`run` is fire-and-forget — it returns `void`. To know when execution completes, listen to the JupyterLab notebook tracker's `executionScheduled` / `executed` events, or use kernel message tracking via `INotebookTracker`.

---
## 9. Reactive Signals — `graphChanged`

The `graphChanged` signal fires every time the user edits the graph in the UI. Use it to synchronise external state with the cell's current graph.

```typescript
chaldene.graphChanged.connect((sender, { cellId, graph }) => {
  console.log(`Cell ${cellId} graph changed:`,
    graph.nodes.length, 'nodes,', graph.edges.length, 'edges');

  // Example: mirror graph to a sidebar panel
  myPanel.update(cellId, graph);
});
```

### Signal timing

| Signal | When it fires | Payload |
|---|---|---|
| `cellReady` | Immediately when VP canvas mounts | `cellId: string` |
| `cellDisposed` | Immediately when VPWidget is disposed | `cellId: string` |
| `graphChanged` | After any interactive graph edit | `{ cellId, graph }` |

All signals are **synchronous** — the handler runs inline, before the emitting call returns.

### Avoiding handler leaks

Always disconnect handlers you no longer need. The idiomatic pattern in a Lumino widget:

```typescript
class MyWidget extends Widget {
  private readonly _chaldene: IChaldeneService;

  constructor(chaldene: IChaldeneService) {
    super();
    this._chaldene = chaldene;
    chaldene.graphChanged.connect(this._onGraphChanged, this);
  }

  dispose(): void {
    this._chaldene.graphChanged.disconnect(this._onGraphChanged, this);
    super.dispose();
  }

  private _onGraphChanged(
    sender: IChaldeneService,
    args: IGraphChangedArgs
  ): void {
    // handle the change
  }
}
```

Passing `this` as the second argument to `connect` / `disconnect` is the Lumino convention for tracking handler context.

---
## 10. Patterns & Best Practices

### Pattern A: Late-caller (cell may not be ready yet)

Your plugin may activate before VP cells have finished mounting. Calling `setGraph` on an unready cell returns `false` and has no effect. Use `cellReady` to queue the update:

```typescript
function applyGraphWhenReady(
  chaldene: IChaldeneService,
  cellId: string,
  graph: IGraph
): void {
  if (chaldene.isCellReady(cellId)) {
    // Cell is already live — apply immediately
    chaldene.setGraph(cellId, graph);
    return;
  }

  // Cell not ready yet — wait for it
  const onReady = (_sender: IChaldeneService, readyId: string) => {
    if (readyId !== cellId) return;
    chaldene.cellReady.disconnect(onReady);  // clean up
    chaldene.setGraph(cellId, graph);
  };
  chaldene.cellReady.connect(onReady);
}
```

### Pattern B: All-cells broadcast

Apply a graph to every VP cell that is currently open, and to any that open later:

```typescript
function broadcastGraph(chaldene: IChaldeneService, graph: IGraph): void {
  // Apply to cells already live
  for (const id of chaldene.getReadyCellIds()) {
    chaldene.setGraph(id, graph);
  }
  // Apply to future cells
  chaldene.cellReady.connect((_sender, cellId) => {
    chaldene.setGraph(cellId, graph);
  });
}
```

### Pattern C: Parameter sweep with execution

Iterate over a list of values, updating a node parameter and running the cell for each:

```typescript
async function sweepThreshold(
  chaldene: IChaldeneService,
  cellId: string,
  thresholds: number[]
): Promise<void> {
  for (const thresh of thresholds) {
    chaldene.setInputValue(cellId, '2', 'in1', thresh);  // node 2, handle in1
    chaldene.run(cellId);
    // Wait for kernel execution to complete before next iteration
    await kernelIdle(notebookPanel);  // your own helper using INotebookTracker
    console.log(`threshold=${thresh} done`);
  }
}
```

### Pattern D: Syncing graph state across cells

Mirror one cell's graph to another whenever it changes:

```typescript
chaldene.graphChanged.connect((_sender, { cellId, graph }) => {
  if (cellId === sourceCellId) {
    // Push a modified copy to the mirror cell
    const mirrored = remapPaths(graph, '/data/mirror/');
    chaldene.setGraph(mirrorCellId, mirrored);
  }
});
```

In [ ]:
# Python illustration of Pattern C: parameter sweep
#
# This shows the logic that your TypeScript code would run — the parameter
# values and graph mutations are identical; only the API calls differ.

import json, copy

# Start from the 3-node pipeline defined earlier
def sweep_threshold(base_graph, threshold_node_id, handle_id, values):
    """
    Yields (threshold, graph) pairs for each threshold value.
    In TypeScript you would call:
        chaldene.setInputValue(cellId, threshold_node_id, handle_id, value)
        chaldene.run(cellId)
    """
    for value in values:
        g = copy.deepcopy(base_graph)
        # Find the node and update the handle
        for node in g['nodes']:
            if node['id'] == str(threshold_node_id):
                for handle in node['data']['inputs']:
                    if handle['id'] == handle_id:
                        handle['defaultValue'] = value
        yield value, g

sweep_values = [0.3, 0.4, 0.5, 0.6, 0.7]

print("Threshold sweep plan (TypeScript equivalent):")
print()
for thresh, g in sweep_threshold(graph, threshold_node_id='2', handle_id='in1',
                                  values=sweep_values):
    node2 = next(n for n in g['nodes'] if n['id'] == '2')
    actual = next(h['defaultValue'] for h in node2['data']['inputs'] if h['id'] == 'in1')
    print(f"  setInputValue(cellId, '2', 'in1', {thresh})  → node in1 = {actual}")
    print(f"  run(cellId)")
    print(f"  await kernelIdle(...)")
    print()

---
## 11. Complete Extension Example

Below is a complete, copy-ready JupyterLab extension that:

1. Declares `IChaldeneService` as a dependency
2. Logs cell lifecycle events
3. Exposes a `chaldene:apply-default-graph` command in the JupyterLab command palette
4. Syncs graph changes to the browser console

```typescript
// src/index.ts
import {
  type JupyterFrontEnd,
  type JupyterFrontEndPlugin
} from '@jupyterlab/application';
import { ICommandPalette } from '@jupyterlab/apputils';
import {
  IChaldeneService,
  type IGraph,
  type IGraphChangedArgs
} from 'chaldene/tokens';

// ── Default graph template ────────────────────────────────────────────────────

function makeDefaultGraph(imagePath: string): IGraph {
  return {
    nodes: [
      {
        id: '0',
        type: 'read_image',
        position: { x: 100, y: 150 },
        data: {
          specName: 'read_image',
          displayLabel: 'read image',
          inputs: [
            {
              id: 'in0', name: 'path', type: 'string',
              displayLabel: 'file', defaultValue: imagePath,
              widget: { type: 'FileInputFromServer',
                        extensions: ['.jpg', '.jpeg', '.png'] }
            },
            {
              id: 'in1', name: 'mode', displayLabel: 'mode',
              defaultValue: 'GRAY',
              widget: { type: 'Dropdown', options: ['GRAY', 'RGB'] }
            }
          ],
          outputs: [
            { id: 'out0', name: 'image', type: 'image', displayLabel: 'image' }
          ]
        }
      },
      {
        id: '1',
        type: 'threshold',
        position: { x: 500, y: 150 },
        data: {
          specName: 'threshold',
          displayLabel: 'threshold',
          inputs: [
            { id: 'in0', name: 'image', type: 'image', displayLabel: 'image' },
            {
              id: 'in1', name: 'thresh', displayLabel: 'threshold',
              defaultValue: 0.5,
              widget: { type: 'Float' }
            }
          ],
          outputs: [
            { id: 'out0', name: 'binary_image',
              type: 'binary image', displayLabel: 'binary image' }
          ]
        }
      }
    ],
    edges: [
      {
        id: '0',
        source: '0', sourceHandle: 'out0',
        target: '1', targetHandle: 'in0'
      }
    ]
  };
}

// ── Plugin ────────────────────────────────────────────────────────────────────

const plugin: JupyterFrontEndPlugin<void> = {
  id: 'my-chaldene-consumer:plugin',
  autoStart: true,
  requires: [IChaldeneService, ICommandPalette],
  activate(
    app: JupyterFrontEnd,
    chaldene: IChaldeneService,
    palette: ICommandPalette
  ): void {

    // ── 1. Lifecycle logging ─────────────────────────────────────────────────
    chaldene.cellReady.connect((_sender, cellId) => {
      console.log('[my-plugin] VP cell ready:', cellId);
    });

    chaldene.cellDisposed.connect((_sender, cellId) => {
      console.log('[my-plugin] VP cell disposed:', cellId);
    });

    // ── 2. React to graph changes ─────────────────────────────────────────────
    chaldene.graphChanged.connect(
      (_sender: IChaldeneService, { cellId, graph }: IGraphChangedArgs) => {
        console.log(`[my-plugin] Graph changed in ${cellId}:`,
          graph.nodes.length, 'nodes');
      }
    );

    // ── 3. Command: apply default graph to the first ready cell ───────────────
    const COMMAND = 'chaldene:apply-default-graph';

    app.commands.addCommand(COMMAND, {
      label: 'Chaldene: Apply Default Graph',
      execute: () => {
        const ids = chaldene.getReadyCellIds();
        if (ids.length === 0) {
          console.warn('[my-plugin] No VP cells are ready');
          return;
        }
        const cellId = ids[0];
        const graph = makeDefaultGraph('/data/sample.png');
        const ok = chaldene.setGraph(cellId, graph);
        if (ok) {
          console.log('[my-plugin] Default graph applied to', cellId);
        }
      }
    });

    palette.addItem({ command: COMMAND, category: 'Chaldene' });

    console.log('[my-plugin] activated. Ready cells:',
      chaldene.getReadyCellIds());
  }
};

export default plugin;
```

---
## 12. API Quick Reference

```typescript
interface IChaldeneService {

  // ── Discovery ──────────────────────────────────────────────────────────────

  /** Returns true if the VP canvas for cellId is mounted and registered. */
  isCellReady(cellId: string): boolean;

  /** Returns the IDs of all currently mounted VP cells. */
  getReadyCellIds(): string[];

  // ── Read ───────────────────────────────────────────────────────────────────

  /**
   * Returns a snapshot of the cell's current graph.
   * Returns undefined if the cell is not ready or has no graph.
   * The returned graph never contains editorContext in node.data.
   */
  getGraph(cellId: string): IGraph | undefined;

  // ── Write ──────────────────────────────────────────────────────────────────

  /**
   * Replaces the entire graph of a VP cell.
   * Returns true on success, false if the cell is not ready.
   * Does NOT arm live auto-execution — call run() explicitly if needed.
   */
  setGraph(cellId: string, graph: IGraph): boolean;

  /**
   * Updates a single input handle's value on an existing graph.
   * Equivalent to the user typing a new value into a node's input widget.
   * Returns true on success, false if the cell is not ready.
   */
  setInputValue(
    cellId: string,
    nodeId: string,
    handleId: string,
    value: unknown
  ): boolean;

  // ── Execution ──────────────────────────────────────────────────────────────

  /**
   * Executes the VP cell (generates code and runs it in the kernel).
   * Equivalent to Shift+Enter on the cell. Fire-and-forget.
   */
  run(cellId: string): void;

  // ── Signals ────────────────────────────────────────────────────────────────

  /** Emits the cellId each time a VP canvas finishes mounting. */
  readonly cellReady: ISignal<IChaldeneService, string>;

  /** Emits the cellId each time a VP cell is deleted or its panel closed. */
  readonly cellDisposed: ISignal<IChaldeneService, string>;

  /**
   * Emits { cellId, graph } whenever the user edits a graph interactively.
   * The emitted graph never contains editorContext.
   */
  readonly graphChanged: ISignal<IChaldeneService, IGraphChangedArgs>;
}
```

### Token import

```typescript
import { IChaldeneService } from 'chaldene/tokens';
// Also available from the same import:
// IGraph, IGraphNode, IGraphEdge, IGraphChangedArgs
```

### Return value contract

| Method | Returns | When `false` / `undefined` |
|---|---|---|
| `isCellReady` | `boolean` | Cell not mounted |
| `getReadyCellIds` | `string[]` | Empty array if none ready |
| `getGraph` | `IGraph \| undefined` | Cell not ready or no graph yet |
| `setGraph` | `boolean` | `false` if cell not ready |
| `setInputValue` | `boolean` | `false` if cell not ready |
| `run` | `void` | No-op if cell not ready |